In [18]:
####################################
#ENVIRONMENT SETUP

In [19]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [20]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [21]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "Radar_ClearAir_Threshold"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/Observation_Data/Radar_Thresholds



In [22]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [23]:
#Load Model Directory Class
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_TRACER = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_Hawaii = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_PRECIP = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Found 193/193 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/WET/MPAS-Model_NSSL/model_run_spinup0hrs/history_cartesian/history.2022-07-01_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/WET/MPAS-Model_NSSL/model_run_spinup0hrs/diag_cartesian/diag.2022-07-01_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           WET
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-07-01 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:193
 # Diag Files:   193
 # Time Steps:   193
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/WET/MPAS-Model_NSSL/model_run_spinup0hrs
 Static File:    TRACER_regional5250_scaled3_

In [24]:
## GetRadarMasks

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

def GetRadarMasks():
    RadarDataMask_PRECIP = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_PRECIP)
    RadarDataMask_TRACER = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_TRACER)
    RadarDataMask_Hawaii = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_Hawaii)
    return RadarDataMask_PRECIP, RadarDataMask_TRACER, RadarDataMask_Hawaii

[RadarDataMask_PRECIP, RadarDataMask_TRACER, RadarDataMask_Hawaii] = GetRadarMasks()

Loaded mask: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/Observation_Data/RadarData/RadarObservationMask/PRECIP_WET_spinup12hrs/RadarObservationMask.nc

Loaded mask: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/Observation_Data/RadarData/RadarObservationMask/TRACER_WET_spinup0hrs/RadarObservationMask.nc

Loaded mask: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/Observation_Data/RadarData/RadarObservationMask/Hawaii_WET_spinup12hrs/RadarObservationMask.nc



In [25]:
###############
#JOB ARRAY SETUP

In [26]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [89]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs
num_jobs = GetNumJobs()

def GetJobInterval(ModelData):    
    JobArray = JobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
    print(f"Running for job_id: {JobArray.job_id}")
    start_job = JobArray.start_job; end_job = JobArray.end_job
    return start_job,end_job

def GetLoopElements(ModelData,start_job,end_job):
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job].tolist()
    return loop_elements

# [start_job,end_job] = GetJobInterval(ModelData)
# loop_elements = GetLoopElements(ModelData,start_job,end_job)

In [28]:
##################
#FUNCTIONS

In [29]:
def CalculateCondensateSpecies(ModelData,t,
                               qt_min=1e-6):
    data = ModelData.GetDataTimestep(t)
    qt = data['qc'] + data['qi'] + data['qr'] + data['qg']
    qtMask = qt > qt_min
    return qt,qtMask

In [30]:
###################################
#PLOTTING FUNCTIONS

In [31]:
## PlotData

COASTLINE_FEATURE = cfeature.COASTLINE
BORDERS_FEATURE   = cfeature.BORDERS
STATES_FEATURE    = cfeature.STATES
# def PlotData(dataArray, levels=15, cmap="viridis"):

#     fig = plt.figure(figsize=(8, 6))
#     ax = plt.axes(projection=ccrs.PlateCarree())

#     cf = ax.contourf(
#         dataArray.longitude,
#         dataArray.latitude,
#         dataArray,#.transpose("latitude", "longitude"),
#         levels=levels,
#         cmap=cmap,
#         transform=ccrs.PlateCarree()
#     )

#     plt.colorbar(cf, ax=ax, label="Range (km)")
#     ax.coastlines()
#     ax.add_feature(cfeature.BORDERS, edgecolor="white",facecolor="none",)
#     ax.add_feature(cfeature.STATES, edgecolor="white",facecolor="none",)

#     plt.tight_layout()
#     return fig, ax

import cartopy.mpl.ticker as cticker

def PlotData(dataArray, levels=15, cmap="viridis"):

    fig = plt.figure(figsize=(8, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())

    cf = ax.contourf(
        dataArray.longitude,
        dataArray.latitude,
        dataArray.T,  # assumes dims (latitude, longitude) OR broadcasting works
        levels=levels,
        cmap=cmap,
        transform=ccrs.PlateCarree()
    )

    plt.colorbar(cf, ax=ax)

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, edgecolor="white", facecolor="none")
    ax.add_feature(cfeature.STATES,  edgecolor="white", facecolor="none")

    # --- Gridlines with labels ---
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.0,   # hide grid lines
        color="none",    # hide grid lines
        alpha=0.0
    )
    
    gl.top_labels = False
    gl.right_labels = False
    
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    plt.tight_layout()
    return fig, ax



In [32]:
def MakeTestPlots(qtMask,
                  ModelData=ModelData_PRECIP,
                  RadarDataMask=RadarDataMask_PRECIP,
                  t=150,zlevel=15):

    refl1 = ModelData.GetDataTimestep_diag(t=t,varName="refl10cm",
                                             printout=False)
    refl1=refl1.where(RadarDataMask==True)
    refl2 = refl1.where(qtMask == True) #*THRESHOLD_TESTING

    #HORIZONTAL
    PlotData(refl1.isel(nVertLevels=zlevel),cmap='turbo')
    PlotData(refl2.isel(nVertLevels=zlevel),cmap='turbo')

    #VERTICAL, LONGITUDE
    a = refl1.mean(dim="latitude")
    b = refl2.mean(dim="latitude")
    
    xr.concat(
        [a, b],
        dim=xr.DataArray(
            ["refl1", "refl2"],
            dims="panel",
            name="panel"
        )
    ).plot(
        col="panel",
        col_wrap=1,
        cmap="turbo",
        figsize=(6, 6)
    )

    #VERTICAL, LATITUDE
    a = refl1.mean(dim="longitude")
    b = refl2.mean(dim="longitude")
    
    xr.concat(
        [a, b],
        dim=xr.DataArray(
            ["refl1", "refl2"],
            dims="panel",
            name="panel"
        )
    ).plot(
        col="panel",
        col_wrap=1,
        cmap="turbo",
        figsize=(6, 6)
    )

In [33]:
###################################
#PLOTTING

In [ ]:
#Load Model Directory Class
Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO= StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

[qt_NSSL_PRECIP,qtMask_NSSL_PRECIP] = CalculateCondensateSpecies(ModelData_NSSL,t=150)
[qt_NSSL_TRACER,qtMask_NSSL_TRACER] = CalculateCondensateSpecies(ModelData_TRACER,t=60)
[qt_NSSL_Hawaii,qtMask_NSSL_Hawaii] = CalculateCondensateSpecies(ModelData_Hawaii,t=100)

In [ ]:
MakeTestPlots(qtMask_NSSL_PRECIP,
              ModelData_PRECIP,
              RadarDataMask_PRECIP,
              t=150)

In [ ]:
MakeTestPlots(qtMask_NSSL_TRACER,
              ModelData_TRACER,
              RadarDataMask_TRACER,
              t=60)

In [ ]:
MakeTestPlots(qtMask_NSSL_Hawaii,
              ModelData_Hawaii,
              RadarDataMask_Hawaii,
              t=100)

In [34]:
###################################
#CALCULATING FUNCTIONS

In [110]:
def LoadModelData(Region,Case,spinup_hours):
    RunType = (Region,Case,"NSSL",spinup_hours)
    ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
    RunType = (Region,Case,"TEMPO",spinup_hours)
    ModelData_TEMPO= StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
    return ModelData_NSSL,ModelData_TEMPO
    
def SaveCondensateThreshold(
    qt_NSSL, qtMask_NSSL,
    qt_TEMPO, qtMask_TEMPO,
    outputPath,timeString):
    """
    Save condensate and masks using the same I/O pattern
    as the working DensityPotentialTemperature script.
    """

    # --- NSSL ---
    # qt_NSSL.to_netcdf(
    #     os.path.join(outputPath, f"qt_NSSL_{timeString}.nc"))
    qtMask_NSSL.to_netcdf(
        os.path.join(outputPath, f"qtMask_NSSL_{timeString}.nc"))

    # --- TEMPO ---
    # qt_TEMPO.to_netcdf(
    #     os.path.join(outputPath, f"qt_TEMPO_{timeString}.nc"))
    qtMask_TEMPO.to_netcdf(
        os.path.join(outputPath, f"qtMask_TEMPO_{timeString}.nc"))
    print(f"Saved to: {outputPath}")
    
def LoadCondensateThreshold(ModelData, t):
    """
    Load condensate mask saved as a single-variable NetCDF file
    (robust single-variable-per-file pattern).
    """

    # codeType = os.path.join("DataAnalysis", "Observation_Data")
    # dataType = "Radar_ClearAir_Threshold"
    # outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
    
    dataDirectory = "/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA"
    outputDirectory_work = os.path.join(dataDirectory,"DataAnalysis","Observation_Data","Radar_ClearAir_Threshold")
    inputPath = os.path.join(outputDirectory_work,f"{Region}_{Case}_{spinup_hours}hrs")

    timeString = ModelData.timeStrings[t]
    filePath = os.path.join(
        inputPath, f"qtMask_{ModelData.mpType}_{timeString}.nc"
    )

    # --- Load safely ---
    qtMask = xr.open_dataset(filePath)["__xarray_dataarray_variable__"]
    return qtMask
#qtMask = LoadCondensateThreshold(ModelData,t)
    
def CalculateCondensateThresholdArray(Region,Case,spinup_hours):
    #Loading Model Classes
    [ModelData_NSSL,ModelData_TEMPO] = LoadModelData(Region,Case,spinup_hours)

    #Loading Data outputPath
    dataDirectory = "/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA"
    outputDirectory_work = os.path.join(dataDirectory,"DataAnalysis","Observation_Data",dataType)
    outputPath = os.path.join(outputDirectory_work,f"{Region}_{Case}_{spinup_hours}hrs")
    os.makedirs(outputPath, exist_ok=True)

    #Job Array
    [start_job,end_job] = GetJobInterval(ModelData_NSSL)
    loop_elements = GetLoopElements(ModelData_NSSL,start_job,end_job)

    #Running
    for t in tqdm(loop_elements):
        timeString = ModelData_NSSL.timeStrings[t]
        
        [qt_NSSL,qtMask_NSSL] = CalculateCondensateSpecies(ModelData_NSSL,t=t)
        [qt_TEMPO,qtMask_TEMPO] = CalculateCondensateSpecies(ModelData_TEMPO,t=t)
        
        SaveCondensateThreshold(
                    qt_NSSL, qtMask_NSSL,
                    qt_TEMPO, qtMask_TEMPO,
                    outputPath,timeString)

In [104]:
###################################
#CALCULATING

In [ ]:
# #Setup
# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# CalculateCondensateThresholdArray(Region,Case,spinup_hours)

In [78]:
[ModelData_NSSL,ModelData_TEMPO] = LoadModelData(Region,Case,spinup_hours)
qtMask = LoadCondensateThreshold(ModelData_NSSL,t=0,mpType="NSSL")

In [ ]:
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
CalculateCondensateThresholdArray(Region,Case,spinup_hours)

In [ ]:
Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
CalculateCondensateThresholdArray(Region,Case,spinup_hours)

In [ ]:
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"
CalculateCondensateThresholdArray(Region,Case,spinup_hours)

In [ ]:
Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
CalculateCondensateThresholdArray(Region,Case,spinup_hours)

In [ ]:
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"
CalculateCondensateThresholdArray(Region,Case,spinup_hours)